In [67]:
import os
from dotenv import load_dotenv

In [68]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


#  Memory Management in Google Agent Developer Kit (ADK)

## Session: Keeping Track of Each Chat

In [ ]:

# Example: Using InMemorySessionService
# This is suitable for local development and testing where data persistence
# across application restarts is not required.
from google.adk.sessions import InMemorySessionService
session_service = InMemorySessionService()

In [ ]:
# Example: Using DatabaseSessionService
# This is suitable for production or development requiring persistent storage.
# You need to configure a database URL (e.g., for SQLite, PostgreSQL, etc.).
# Requires: pip install google-adk[sqlalchemy] and a database driver (e.g., psycopg2 for PostgreSQL)
from google.adk.sessions import DatabaseSessionService
# Example using a local SQLite file:
db_url = "sqlite:///./my_agent_data.db"
session_service = DatabaseSessionService(db_url=db_url)

In [ ]:

# Example: Using VertexAiSessionService
# This is suitable for scalable production on Google Cloud Platform, leveraging
# Vertex AI infrastructure for session management.
# Requires: pip install google-adk[vertexai] and GCP setup/authentication
from google.adk.sessions import VertexAiSessionService

PROJECT_ID = "your-gcp-project-id" # Replace with your GCP project ID
LOCATION = "us-central1" # Replace with your desired GCP location
# The app_name used with this service should correspond to the Reasoning Engine ID or name
REASONING_ENGINE_APP_NAME = "projects/your-gcp-project-id/locations/us-central1/reasoningEngines/your-engine-id" # Replace with your Reasoning Engine resource name

session_service = VertexAiSessionService(project=PROJECT_ID, location=LOCATION)
# When using this service, pass REASONING_ENGINE_APP_NAME to service methods:
# session_service.create_session(app_name=REASONING_ENGINE_APP_NAME, ...)
# session_service.get_session(app_name=REASONING_ENGINE_APP_NAME, ...)
# session_service.append_event(session, event, app_name=REASONING_ENGINE_APP_NAME)
# session_service.delete_session(app_name=REASONING_ENGINE_APP_NAME, ...)
     

## State: The Session’s Scratchpad

In [69]:
# Import necessary classes from the Google Agent Developer Kit (ADK)
from google.adk.agents import LlmAgent
from google.adk.sessions import InMemorySessionService, Session
from google.adk.runners import Runner
from google.genai.types import Content, Part, GenerateContentConfig, AutomaticFunctionCallingConfig

# Define an LlmAgent with an output_key.
greeting_agent = LlmAgent(
    name="Greeter",
    model="gemini-flash-latest",  # the book's gemini-2.0-flash is retired
    instruction="Generate a short, friendly greeting.",
    output_key="last_greeting",
    # ADK runs its own tool loop and never uses the genai SDK's automatic function
    # calling, but leaves it enabled, so the SDK logs a one-time "Direct use of AFC
    # ... is not recommended" warning. Disabling it here removes the warning.
    generate_content_config=GenerateContentConfig(
        automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
    ),
)

# --- Setup Runner and Session ---
app_name, user_id, session_id = "state_app", "user1", "session1"
session_service = InMemorySessionService()
runner = Runner(
    agent=greeting_agent,
    app_name=app_name,
    session_service=session_service
)
# create_session is a coroutine: without the await, `session` is the coroutine object.
session = await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)

print(f"Initial state: {session.state}")

# --- Run the Agent ---
user_message = Content(role="user", parts=[Part(text="Hello")])
print("\n--- Running the agent ---")
async for event in runner.run_async(
    user_id=user_id,
    session_id=session_id,
    new_message=user_message
):
    if event.is_final_response():
      print("Agent responded.")

# --- Check Updated State ---
# Correctly check the state *after* the runner has finished processing all events.
# get_session is also a coroutine, and its arguments are keyword-only.
updated_session = await session_service.get_session(
    app_name=app_name, user_id=user_id, session_id=session_id
)
print(f"\nState after agent run: {updated_session.state}")

Initial state: {}

--- Running the agent ---
Agent responded.

State after agent run: {'last_greeting': "Hello! It's wonderful to meet you. How can I help you today?"}


## Google ADK output_key + Ollama

In [70]:
# Same output_key example, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b

import os
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai.types import Content, Part

# LiteLLM needs to know where the local Ollama server listens.
os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

# Same agent, local model. No generate_content_config: that is a Gemini SDK setting,
# and the AFC warning it suppressed comes from the Gemini SDK, which is not used here.
ollama_greeting_agent = LlmAgent(
    name="Greeter",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction="Generate a short, friendly greeting.",
    output_key="last_greeting"
)

# --- Setup Runner and Session ---
app_name, user_id, session_id = "state_app_ollama", "user1", "session1"
session_service = InMemorySessionService()
runner = Runner(
    agent=ollama_greeting_agent,
    app_name=app_name,
    session_service=session_service
)
session = await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id
)

print(f"Initial state: {session.state}")

# --- Run the Agent ---
print("\n--- Running the agent ---")
async for event in runner.run_async(
    user_id=user_id,
    session_id=session_id,
    new_message=Content(role="user", parts=[Part(text="Hello")])
):
    if event.is_final_response():
      print("Agent responded.")

# --- Check Updated State ---
updated_session = await session_service.get_session(
    app_name=app_name, user_id=user_id, session_id=session_id
)
print(f"\nState after agent run: {updated_session.state}")

Initial state: {}

--- Running the agent ---
Agent responded.

State after agent run: {'last_greeting': 'Hello there! How can I assist you today?'}


In [5]:
import os
import time
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.tool_context import ToolContext
from google.genai.types import Content, Part, GenerateContentConfig, AutomaticFunctionCallingConfig

# --- Define the Recommended Tool-Based Approach ---
def log_user_login(tool_context: ToolContext) -> dict:
    """
    Updates the session state upon a user login event.
    This tool encapsulates all state changes related to a user login.
    Args:
        tool_context: Automatically provided by ADK, gives access to session state.
    Returns:
        A dictionary confirming the action was successful.
    """
    # Access the state directly through the provided context.
    state = tool_context.state

    # Get current values or defaults, then update the state.
    # This is much cleaner and co-locates the logic.
    login_count = state.get("user:login_count", 0) + 1
    state["user:login_count"] = login_count
    state["task_status"] = "active"
    state["user:last_login_ts"] = time.time()
    state["temp:validation_needed"] = True

    print("State updated from within the `log_user_login` tool.")

    return {
        "status": "success",
        "message": f"User login tracked. Total logins: {login_count}."
    }

# --- Demonstration of Usage ---
# The book simulates a direct call by hand-building a ToolContext. That no longer
# constructs (ToolContext is now an alias of Context, and InvocationContext is not
# exported from tool_context), and a hand-made context would only hold a state delta
# that nothing commits to the session. So the tool is run the way ADK runs tools: an
# agent decides to call it, and the Runner builds the ToolContext and commits the state.

# 1. Setup
session_service = InMemorySessionService()
app_name, user_id, session_id = "state_app_tool", "user3", "session3"
# create_session is a coroutine: without the await, `session` is the coroutine object.
session = await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id,
    state={"user:login_count": 0, "task_status": "idle"}
)
print(f"Initial state: {session.state}")

# ADK 2.7.1 builds tool declarations with an experimental feature that is on by default,
# and warns once per process that it is. This switches that feature off so no
# warning is printed; ADK then falls back to its older function-declaration builder.
os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")

# 2. An agent that owns the tool
login_agent = LlmAgent(
    name="LoginTracker",
    model="gemini-flash-latest",
    instruction="When the user says they have logged in, call the log_user_login tool.",
    tools=[log_user_login],
    generate_content_config=GenerateContentConfig(
        automatic_function_calling=AutomaticFunctionCallingConfig(disable=True)
    ),
)
runner = Runner(agent=login_agent, app_name=app_name, session_service=session_service)

# 3. Run it, so the Runner calls the tool with a real ToolContext
async for event in runner.run_async(
    user_id=user_id,
    session_id=session_id,
    new_message=Content(role="user", parts=[Part(text="I just logged in.")]),
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"Agent: {event.content.parts[0].text}")

# 4. Check the updated state
# get_session is also a coroutine, and its arguments are keyword-only.
updated_session = await session_service.get_session(
    app_name=app_name, user_id=user_id, session_id=session_id
)
print(f"State after tool execution: {updated_session.state}")

# The temp: key is missing from the stored state on purpose: ADK never persists
# temp:-prefixed keys, so it is visible inside the tool and gone afterwards.

Initial state: {'task_status': 'idle', 'user:login_count': 0}


State updated from within the `log_user_login` tool.


Agent: Welcome back! Your login has been recorded. Total logins: 1.
State after tool execution: {'task_status': 'active', 'user:login_count': 1, 'user:last_login_ts': 1789252843.924533}


## Google ADK ToolContext + Ollama

In [71]:
# Same tool-based state update, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
# Reuses log_user_login from the cell above.

import os
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
# Use the 'ollama_chat' provider, not 'ollama': it is the one that handles tool calling
# correctly. A no-argument tool call is well within what qwen2.5:7b manages.
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

# 1. Setup
session_service = InMemorySessionService()
app_name, user_id, session_id = "state_app_tool_ollama", "user3", "session3"
session = await session_service.create_session(
    app_name=app_name,
    user_id=user_id,
    session_id=session_id,
    state={"user:login_count": 0, "task_status": "idle"}
)
print(f"Initial state: {session.state}")

# 2. An agent that owns the tool
ollama_login_agent = LlmAgent(
    name="LoginTracker",
    model=LiteLlm(model=OLLAMA_MODEL),
    instruction="When the user says they have logged in, call the log_user_login tool.",
    tools=[log_user_login],
)
runner = Runner(agent=ollama_login_agent, app_name=app_name, session_service=session_service)

# 3. Run it, so the Runner calls the tool with a real ToolContext
async for event in runner.run_async(
    user_id=user_id,
    session_id=session_id,
    new_message=Content(role="user", parts=[Part(text="I just logged in.")]),
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"Agent: {event.content.parts[0].text}")

# 4. Check the updated state
updated_session = await session_service.get_session(
    app_name=app_name, user_id=user_id, session_id=session_id
)
print(f"State after tool execution: {updated_session.state}")

Initial state: {'task_status': 'idle', 'user:login_count': 0}
State updated from within the `log_user_login` tool.
Agent: Great! It looks like you've logged in successfully. This is your first login.
State after tool execution: {'task_status': 'active', 'user:login_count': 1, 'user:last_login_ts': 1789253313.722115}


## Memory: Long-Term Knowledge with MemoryService

In [ ]:

# Example: Using InMemoryMemoryService
# This is suitable for local development and testing where data persistence
# across application restarts is not required. Memory content is lost when the app stops.
from google.adk.memory import InMemoryMemoryService
memory_service = InMemoryMemoryService()

In [ ]:

# Example: Using VertexAiRagMemoryService
# This is suitable for scalable production on Google Cloud Platform, leveraging
# Vertex AI RAG (Retrieval Augmented Generation) for persistent, searchable memory.
# Requires: pip install google-adk[vertexai], GCP setup/authentication, and a Vertex AI RAG Corpus.
from google.adk.memory import VertexAiRagMemoryService

# The resource name of your Vertex AI RAG Corpus
RAG_CORPUS_RESOURCE_NAME = "projects/your-gcp-project-id/locations/us-central1/ragCorpora/your-corpus-id" # Replace with your Corpus resource name

# Optional configuration for retrieval behavior
SIMILARITY_TOP_K = 5 # Number of top results to retrieve
VECTOR_DISTANCE_THRESHOLD = 0.7 # Threshold for vector similarity

memory_service = VertexAiRagMemoryService(
    rag_corpus=RAG_CORPUS_RESOURCE_NAME,
    similarity_top_k=SIMILARITY_TOP_K,
    vector_distance_threshold=VECTOR_DISTANCE_THRESHOLD
)
# When using this service, methods like add_session_to_memory and search_memory
# will interact with the specified Vertex AI RAG Corpus.

#  Memory Management in LangChain and LangGraph

In [73]:
# langchain.memory was removed in LangChain 1.x. The same class now lives in
# langchain_core under a new name; its add_user_message / add_ai_message API is unchanged.
from langchain_core.chat_history import InMemoryChatMessageHistory

# Initialize the history object
history = InMemoryChatMessageHistory()

# Add user and AI messages
history.add_user_message("I'm heading to New York next week.")
history.add_ai_message("Great! It's a fantastic city.")

# Access the list of messages
print(history.messages)

[HumanMessage(content="I'm heading to New York next week.", additional_kwargs={}, response_metadata={}), AIMessage(content="Great! It's a fantastic city.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [74]:
# ConversationBufferMemory was removed with langchain.memory in LangChain 1.x and has
# no successor in langchain_core: the legacy memory classes moved to the separate
# langchain-classic package, and chat history is the recommended replacement. What the
# class did is two lines - store the turn as messages, then render them as one string.
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import get_buffer_string

# Initialize memory
memory = InMemoryChatMessageHistory()

# Save a conversation turn (memory.save_context({"input": ...}, {"output": ...}))
memory.add_user_message("What's the weather like?")
memory.add_ai_message("It's sunny today.")

# Load the memory as a string (memory.load_memory_variables({}))
print({"history": get_buffer_string(memory.messages)})

{'history': "Human: What's the weather like?\nAI: It's sunny today."}


In [75]:
# Three of the book's four imports are gone in LangChain 1.x: langchain.chains (LLMChain),
# langchain.prompts (moved to langchain_core) and langchain.memory. A chain is now
# prompt | llm, and what LLMChain did with its memory - read the history into the prompt,
# call the model, save the turn - is written out in predict() below.
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.messages import get_buffer_string

# 1. Define LLM and Prompt
# The book's OpenAI(...) is the legacy completions endpoint; use the chat model.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
template = """You are a helpful travel agent.

Previous conversation:
{history}

New question: {question}
Response:"""
prompt = PromptTemplate.from_template(template)

# 2. Configure Memory
memory = InMemoryChatMessageHistory()

# 3. Build the Chain
conversation = prompt | llm | StrOutputParser()


def predict(question: str) -> str:
    """One turn: render the history into the prompt, call the model, save both messages."""
    response = conversation.invoke(
        {"history": get_buffer_string(memory.messages), "question": question}
    )
    memory.add_user_message(question)
    memory.add_ai_message(response)
    return response


# 4. Run the Conversation
response = predict("I want to book a flight.")
print(response)
response = predict("My name is Sam, by the way.")
print(response)
response = predict("What was my name again?")
print(response)

Sure! I can help you with that. Could you please provide me with the following details?

1. Departure city and airport
2. Destination city and airport
3. Travel dates (departure and return, if applicable)
4. Number of passengers
5. Any specific preferences (e.g., direct flights, preferred airlines, class of service)

Once I have this information, I can assist you in finding the best flight options!
Nice to meet you, Sam! Now that I have your name, could you please provide me with the flight details I mentioned earlier? That way, I can help you find the best options for your trip!
Your name is Sam!


## LangChain string memory + Ollama

In [76]:
# Same string-history chain, running on a local model served by Ollama.
# Prerequisites:
#   ollama pull llama3.1:8b
# Reuses prompt and predict from the cell above: predict reads the module-level
# `conversation` and `memory`, so rebinding those two is the whole port.
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory

OLLAMA_MODEL = "llama3.1:8b"  # any model from `ollama list`, e.g. qwen2.5:7b

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
memory = InMemoryChatMessageHistory()
conversation = prompt | llm | StrOutputParser()

response = predict("I want to book a flight.")
print(response)
response = predict("My name is Sam, by the way.")
print(response)
response = predict("What was my name again?")
print(response)

I'd be happy to help you book a flight. Can you please tell me where you're planning to go and when you'd like to travel? Are you looking for a specific airline or would you like me to suggest some options?
Nice to meet you, Sam! I'm glad you're planning a trip. I've taken note of your interest in booking a flight. Now, let's get started on the details. Where are you thinking of traveling to, and when would you like to depart?
Your name is Sam!


In [77]:
# LLMChain and ConversationBufferMemory are gone in LangChain 1.x (see the two cells
# above). The one difference from the previous cell: return_messages=True meant the
# history was injected as a list of messages, so here the MessagesPlaceholder is filled
# with memory.messages directly instead of a rendered string.
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

# 1. Define Chat Model and Prompt
llm = ChatOpenAI(model="gpt-4o-mini")
prompt = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template("You are a friendly assistant."),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessagePromptTemplate.from_template("{question}")
    ]
)

# 2. Configure Memory
memory = InMemoryChatMessageHistory()

# 3. Build the Chain
conversation = prompt | llm | StrOutputParser()


def predict(question: str) -> str:
    """One turn: fill the placeholder with the messages so far, call the model, save the turn."""
    response = conversation.invoke({"chat_history": memory.messages, "question": question})
    memory.add_user_message(question)
    memory.add_ai_message(response)
    return response


# 4. Run the Conversation
response = predict("Hi, I'm Jane.")
print(response)
response = predict("Do you remember my name?")
print(response)

Hi Jane! How can I assist you today?
Yes, your name is Jane! How can I help you today?


## LangChain message memory + Ollama

In [12]:
# Same message-history chain, running on a local model served by Ollama.
# Prerequisites:
#   ollama pull llama3.1:8b
# Reuses prompt and predict from the cell above.
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory

OLLAMA_MODEL = "llama3.1:8b"

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
memory = InMemoryChatMessageHistory()
conversation = prompt | llm | StrOutputParser()

response = predict("Hi, I'm Jane.")
print(response)
response = predict("Do you remember my name?")
print(response)

Hi Jane! It's nice to meet you. Is there something I can help you with today?


I should be able to remember it since you just told me. Your name is Jane. How can I assist you today, Jane?


In [13]:
# The book calls this listing pseudo-code: State, BaseStore, prompt_template and llm are
# never defined, and nothing runs the two nodes. Here it is made real. The store holds
# the agent's instructions (procedural memory); call_model answers with them, then
# update_instructions reflects on the conversation and rewrites them.
#
# Three fixes inside the book's nodes: it wrote to ("agent_instructions",) but searched
# ("instructions",); store.get returns one Item, not a list, so the [0] is gone; and
# llm.invoke returns a message, not a dict, so the reflection uses structured output.
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
State = MessagesState


class NewInstructions(TypedDict):
    new_instructions: str


prompt_template = (
    "Here are your current instructions:\n{instructions}\n\n"
    "Here is the conversation you just had:\n{conversation}\n\n"
    "Rewrite the instructions in two sentences so that you serve this user better next time."
)


# Node that uses the instructions to generate a response
def call_model(state: State, store: BaseStore):
    namespace = ("agent_instructions",)
    # Retrieve the latest instructions from the store
    instructions = store.get(namespace, key="agent_a")

    # Use the retrieved instructions as the system prompt
    response = llm.invoke(
        [{"role": "system", "content": instructions.value["instructions"]}] + state["messages"]
    )
    return {"messages": [response]}


# Node that updates the agent's instructions
def update_instructions(state: State, store: BaseStore):
    namespace = ("agent_instructions",)
    # Get the current instructions from the store
    current_instructions = store.search(namespace)[0]

    # Create a prompt to ask the LLM to reflect on the conversation
    # and generate new, improved instructions
    prompt = prompt_template.format(
        instructions=current_instructions.value["instructions"],
        conversation=state["messages"],
    )

    # Get the new instructions from the LLM
    output = llm.with_structured_output(NewInstructions).invoke(prompt)
    new_instructions = output["new_instructions"]

    # Save the updated instructions back to the store
    store.put(namespace, "agent_a", {"instructions": new_instructions})


# Wire the two nodes into a graph. Nodes that declare a `store` parameter get the
# store passed to compile() injected at run time.
graph = (
    StateGraph(State)
    .add_node("call_model", call_model)
    .add_node("update_instructions", update_instructions)
    .add_edge(START, "call_model")
    .add_edge("call_model", "update_instructions")
    .add_edge("update_instructions", END)
)

store = InMemoryStore()
store.put(("agent_instructions",), "agent_a", {"instructions": "You are a helpful assistant."})
app = graph.compile(store=store)

print("Instructions before:", store.get(("agent_instructions",), "agent_a").value["instructions"])

result = app.invoke(
    {"messages": [{"role": "user", "content": "Explain recursion. Keep it short, I am a Python beginner and I dislike jargon."}]}
)
print("\nAgent:", result["messages"][-1].content)

print("\nInstructions after:", store.get(("agent_instructions",), "agent_a").value["instructions"])

Instructions before: You are a helpful assistant.



Agent: Recursion is a way of solving problems where a function calls itself to break the problem into smaller parts. 

Think of it like this: if you want to find the factorial of a number (like 5!), you can say:

- 5! = 5 × 4!
- 4! = 4 × 3!
- 3! = 3 × 2!
- 2! = 2 × 1!
- 1! = 1 (this is the base case that stops the recursion)

In Python, a simple recursive function for factorial looks like this:

```python
def factorial(n):
    if n == 1:  # Base case
        return 1
    else:
        return n * factorial(n - 1)  # Recursive call
```

You start with a big problem (like 5!) and break it down until you reach a simple case (1!). Then you build back up to get the answer.

Instructions after: Provide clear and concise explanations without jargon, especially for beginners. Use relatable examples to illustrate concepts, making them easier to understand.


## LangGraph + Ollama

In [14]:
# Same procedural-memory graph, running on a local model served by Ollama.
# Prerequisites:
#   ollama pull llama3.1:8b
# Reuses the nodes and graph from the cell above. Both nodes read the module-level
# `llm` when they run, so rebinding it is the whole port; a fresh store and a fresh
# compile keep this run separate from the hosted one.
from langchain_ollama import ChatOllama
from langgraph.store.memory import InMemoryStore

OLLAMA_MODEL = "llama3.1:8b"

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

store = InMemoryStore()
store.put(("agent_instructions",), "agent_a", {"instructions": "You are a helpful assistant."})
app = graph.compile(store=store)

print("Instructions before:", store.get(("agent_instructions",), "agent_a").value["instructions"])

result = app.invoke(
    {"messages": [{"role": "user", "content": "Explain recursion. Keep it short, I am a Python beginner and I dislike jargon."}]}
)
print("\nAgent:", result["messages"][-1].content)

print("\nInstructions after:", store.get(("agent_instructions",), "agent_a").value["instructions"])

Instructions before: You are a helpful assistant.



Agent: Recursion is a way to solve problems by breaking them down into smaller versions of the same problem. Here's a simple example:

Imagine you have a stack of boxes, and you want to count how many boxes are in the stack. You can count the top box, and then ask yourself, "How many boxes are in the stack below this one?" You can count those, and then ask the same question again, and again, until you reach the bottom of the stack.

In Python, you can write a function that does the same thing. It calls itself, over and over, until it reaches the "bottom" of the problem. Here's a simple example:

```python
def count_boxes(box):
    if box == 0:  # base case: when there are no more boxes
        return 0
    else:
        return 1 + count_boxes(box - 1)  # call itself with one less box

print(count_boxes(5))  # prints 5
```

In this example, the function `count_boxes` calls itself with one less box each time, until it reaches the base case (when there are no more boxes).

Instructions a

In [15]:
from langgraph.store.memory import InMemoryStore

# A placeholder for a real embedding function
def embed(texts: list[str]) -> list[list[float]]:
    # In a real application, use a proper embedding model
    return [[1.0, 2.0] for _ in texts]

# Initialize an in-memory store. For production, use a database-backed store.
store = InMemoryStore(index={"embed": embed, "dims": 2})

# Define a namespace for a specific user and application context
user_id = "my-user"
application_context = "chitchat"
namespace = (user_id, application_context)

# 1. Put a memory into the store
store.put(
    namespace,
    "a-memory",  # The key for this memory
    {
        "rules": [
            "User likes short, direct language",
            "User only speaks English & python",
        ],
        "my-key": "my-value",
    },
)

# 2. Get the memory by its namespace and key
item = store.get(namespace, "a-memory")
print("Retrieved Item:", item)

# 3. Search for memories within the namespace, filtering by content
# and sorting by vector similarity to the query.
items = store.search(
    namespace,
    filter={"my-key": "my-value"},
    query="language preferences"
)
print("Search Results:", items)


Retrieved Item: Item(namespace=['my-user', 'chitchat'], key='a-memory', value={'rules': ['User likes short, direct language', 'User only speaks English & python'], 'my-key': 'my-value'}, created_at='2026-09-12T22:41:08.854480+00:00', updated_at='2026-09-12T22:41:08.854483+00:00')
Search Results: [Item(namespace=['my-user', 'chitchat'], key='a-memory', value={'rules': ['User likes short, direct language', 'User only speaks English & python'], 'my-key': 'my-value'}, created_at='2026-09-12T22:41:08.854480+00:00', updated_at='2026-09-12T22:41:08.854483+00:00', score=0.9999999999999998)]
